# 3.4 Nhãn nhiễu (CIFAR-10, PreAct ResNet-18) trên Kaggle

Chạy 4 cấu hình tốt nhất của Bảng 2 (ERM, ERM + dropout, mixup, mixup + dropout) cho **một mức
nhiễu mỗi phiên** (`NOISE`), rồi gom bảng so với bài báo. Mỗi lần chạy 200 epoch ≈ 1,5–2 giờ GPU,
nên một mức nhiễu (4 lần) vừa một phiên 12 giờ. Có 2 GPU (T4 x2) thì chạy 2 lần song song.

**Trước khi chạy** (xem `kaggle/README.md`):
1. Accelerator: **GPU P100** hoặc T4. Internet: **On**.
2. Add Input → dataset chứa thư mục `Source`.
3. Chạy tiếp phiên bị ngắt: Add Input → **output của version cũ**; ô setup tự chép `results/exp34_noise*` về.
4. Chạy: chỉnh `NOISE` → **Save Version → Save & Run All (Commit)**. Phiên tự dừng ở ~11,3 giờ nếu chưa xong.

In [ ]:
# ==== Tham số ====
NOISE = 0.5        # mức nhiễu của phiên này: 0.2 / 0.5 / 0.8, hoặc danh sách vd [0.2, 0.5]
SEEDS = [0]        # thêm 1, 2 nếu còn quota
SMOKE = False      # True: chạy thử 1 epoch ngắn + kiểm tra resume (vài phút)
RUN_FULL = True    # chạy thật 4 cấu hình x len(SEEDS), 200 epoch
AMP = True         # mixed precision (bài báo FP32; ghi rõ trong báo cáo)
WORKERS = 4        # tổng số nhân CPU, chia đều cho các GPU
TIME_BUDGET_H = 11.3  # Kaggle cắt phiên ở 12 giờ: dừng trước để output (last.pt) vẫn được lưu

In [ ]:
import os, sys, shutil, glob, subprocess, json, time, shlex
T0 = time.time()
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CHƯA BẬT GPU")

cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/core/mixup.py', recursive=True)]
assert cands, "Không thấy Source. Add Input -> chọn dataset chứa thư mục Source"
# Output phiên trước cũng có bản sao Source (cũ, nằm cạnh results/): ưu tiên Source của dataset
cands.sort(key=lambda c: os.path.exists(os.path.join(os.path.dirname(c), 'results')))
print("Các Source tìm thấy:", cands)
SRC = '/kaggle/working/Source'
if not os.path.exists(SRC):
    shutil.copytree(cands[0], SRC)
os.chdir(SRC)
sys.path.insert(0, os.path.join(SRC, 'experiments'))   # để import exp34_noisy, exp34_report
DATA = '/kaggle/working/data'
OUT = '/kaggle/working/results'

# Chép kết quả dở dang của phiên trước (nếu đã Add Input output cũ) để resume
for ck in glob.glob('/kaggle/input/**/results/exp34_noise*/*/last.pt', recursive=True):
    run_dir = os.path.dirname(ck)
    dst = os.path.join(OUT, os.path.relpath(run_dir, run_dir.split('/results/')[0] + '/results'))
    if not os.path.exists(dst):
        shutil.copytree(run_dir, dst)
        print("khôi phục", dst)

# CIFAR-10: chép từ input nếu có (vd output phiên trước) thay vì tải ~170MB rất chậm từ Toronto
src_cifar = glob.glob('/kaggle/input/**/cifar-10-batches-py', recursive=True)
if src_cifar and not os.path.exists(f"{DATA}/cifar-10-batches-py"):
    shutil.copytree(src_cifar[0], f"{DATA}/cifar-10-batches-py")
    print("chép CIFAR-10 từ", src_cifar[0])

# Tải CIFAR-10 trước, để 2 tiến trình chạy song song không cùng tải/giải nén vào một chỗ
import torchvision
torchvision.datasets.CIFAR10(DATA, train=True, download=True)
torchvision.datasets.CIFAR10(DATA, train=False, download=True)

In [ ]:
!python -m pytest tests -q

In [ ]:
import threading, queue
NUM_GPUS = max(1, torch.cuda.device_count())   # T4 x2 -> 2 lần chạy song song
DEADLINE = T0 + TIME_BUDGET_H * 3600
STOP = threading.Event()
_lock = threading.Lock()

def say(msg):
    with _lock:
        print(msg.rstrip(chr(10)), flush=True)

def run(args, gpu=0, workers=WORKERS):
    # Chạy 1 lần train trên GPU `gpu`. True = xong; False = bị dừng vì hết giờ (last.pt đã lưu mỗi epoch).
    cmd = [sys.executable, "experiments/exp34_noisy.py", "--data-dir", DATA, "--out-dir", OUT,
           "--workers", str(workers)] + (["--amp"] if AMP else []) + shlex.split(args)
    tag = f"[gpu{gpu}] "
    say(tag + "$ " + " ".join(cmd))
    t = time.time()
    env = {**os.environ, "PYTHONUNBUFFERED": "1", "CUDA_VISIBLE_DEVICES": str(gpu)}
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in proc.stdout:
        say(tag + line)
        if time.time() > DEADLINE or STOP.is_set():
            STOP.set()
            proc.kill()
            proc.wait()
            say(tag + "-> HẾT GIỜ: dừng, phiên sau Add Input output này để chạy tiếp")
            return False
    rc = proc.wait()
    say(f"{tag}-> xong sau {time.time()-t:.0f}s, exit={rc}")
    if rc != 0:
        raise RuntimeError(f"{tag}lệnh lỗi (exit={rc}): {' '.join(cmd)}")
    return True

def run_all(jobs):
    # Chia các lần chạy cho NUM_GPUS GPU qua một hàng đợi; CPU chia đều cho các tiến trình.
    q = queue.Queue()
    for j in jobs:
        q.put(j)
    errors = []
    def worker(gpu):
        while not STOP.is_set():
            try:
                args = q.get_nowait()
            except queue.Empty:
                return
            try:
                run(args, gpu, max(1, WORKERS // NUM_GPUS))
            except Exception as e:      # lỗi 1 lần chạy không chặn GPU còn lại
                errors.append(e)
                say(f"LỖI: {e}")
    threads = [threading.Thread(target=worker, args=(g,)) for g in range(NUM_GPUS)]
    for th in threads:
        th.start()
    for th in threads:
        th.join()
    return errors

## 1. Chạy thử

mixup + dropout, 1 epoch × 20 batch, kiểm tra `summary.json` có lỗi train trên nhãn thật/nhiễu
và tỉ lệ nhãn sai ≈ NOISE·0.9. Sau đó chạy lại với 2 epoch để kiểm tra resume.

In [ ]:
import pandas as pd
if SMOKE:
    base = f"--noise-rate {NOISE} --alpha 8 --dropout 0.3 --limit-batches 20 --run-name smoke34"
    assert run(base + " --epochs 1 --no-resume")
    s = json.load(open(f"{OUT}/smoke34/summary.json"))
    print({k: s[k] for k in ["train_clean_err", "train_real_err", "train_corrupted_err", "noise_frac_selected", "noise_frac_wrong"]})
    assert abs(s["noise_frac_wrong"] - NOISE * 0.9) < 0.02, "Tỉ lệ nhãn sai không khớp NOISE·0.9"
    run(base + " --epochs 2")
    h = pd.read_csv(f"{OUT}/smoke34/history.csv")
    assert list(h.epoch) == [0, 1], "Resume sai"
    print("Smoke + resume OK")

## 2. Chạy thật

Lần chạy nào đã có `summary.json` đầy đủ thì bỏ qua; lần nào dở (có `last.pt`) thì tự chạy tiếp.

In [ ]:
from exp34_report import PAPER
from exp34_noisy import method_name

NOISES = NOISE if isinstance(NOISE, (list, tuple)) else [NOISE]
configs = [k for k in PAPER if any(abs(k[0] - n) < 1e-9 for n in NOISES)]
print("Cấu hình (nhiễu, α, p):", configs, "| seeds:", SEEDS)
todo = []
for seed in SEEDS:
    for rate, alpha, p in configs:
        name = f"exp34_noise{round(rate * 100)}/{method_name(alpha, p)}_seed{seed}"
        sp = f"{OUT}/{name}/summary.json"
        # đã train xong nhưng thiếu train_clean_err (bản cũ) -> chạy lại: chỉ nạp last.pt và đo lại
        if os.path.exists(sp) and "train_clean_err" in json.load(open(sp)):
            print("đã xong, bỏ qua:", name)
        else:
            todo.append((name, f"--noise-rate {rate:g} --alpha {alpha:g} --dropout {p:g} --seed {seed} --epochs 200"))
if RUN_FULL:
    print("Số GPU:", NUM_GPUS)
    errors = run_all([args for _, args in todo])
    left = [n for n, _ in todo
            if not (os.path.exists(f"{OUT}/{n}/summary.json") and "train_clean_err" in json.load(open(f"{OUT}/{n}/summary.json")))]
    print("CÒN DỞ (chạy tiếp ở phiên sau):" if left else "XONG HẾT mức nhiễu", NOISES, left)
    assert not errors, errors
else:
    print("Bỏ qua chạy thật (RUN_FULL = False)")

## 3. Bảng so với bài báo + biểu đồ lỗi test theo epoch

In [ ]:
from IPython.display import Markdown, Image, display
!python experiments/exp34_report.py --results {OUT}
if os.path.exists(f"{OUT}/exp34_report.md"):
    display(Markdown(open(f"{OUT}/exp34_report.md", encoding="utf-8").read()))
if os.path.exists(f"{OUT}/exp34_curves.png"):
    display(Image(f"{OUT}/exp34_curves.png"))

In [ ]:
# Đóng gói log (không kèm .pt) để tải về; last.pt vẫn nằm trong Output để resume phiên sau
!cd /kaggle/working && zip -qr results_logs.zip results -x '*.pt' && ls -lh results_logs.zip